open balanced dataset and start set up tokenizer

In [ ]:
#!pip install datasets; evaluate; numpy; sklearn; matplotlib; google.colab

In [ ]:
from datasets import load_dataset,ClassLabel, Dataset

from transformers import Trainer,AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,EarlyStoppingCallback
import numpy as np
from evaluate import load
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
dataset = load_dataset("csv", data_files="/content/drive/MyDrive/lyricsclassificationdata/archive/balancedSubset10kClassicalCountryHipHopElectronic.csv")

In [ ]:
genres = list(set(dataset["train"]["genre"]))
class_label = ClassLabel(names=genres)
dataset = dataset["train"].cast_column("genre", class_label)

In [ ]:
dataset_split : Dataset = dataset.train_test_split(test_size=0.2, seed=69, stratify_by_column="genre")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

LITTLE SETUP

In [ ]:
def tokenize_function (set : Dataset) :
    return tokenizer(set['lyrics'], padding="max_length", truncation=True,max_length=512)

In [ ]:
tokenized_dataset = dataset_split.map(tokenize_function, batched=True)

In [ ]:
tokenized_dataset = tokenized_dataset.rename_column("genre", "labels")

In [ ]:

model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

In [ ]:
metric_f1 = load("f1")
metric_acc = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": metric_acc.compute(
            predictions=preds,
            references=labels
        )["accuracy"],

        "f1_macro": metric_f1.compute(
            predictions=preds,
            references=labels,
            average="macro"  
        )["f1"],
    }

In [ ]:
#look at medium article for this 

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",       
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    label_smoothing_factor=0.05,
    fp16=True,
)

In [ ]:
trainer = Trainer(
    model=model,                        
    args=training_args,                 
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],   
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()
trainer.save_model("models/model10kClassicalCountryHipHopElectronic")

In [ ]:
#confusion matrix to see to understand why model is so shit

predictions = trainer.predict(tokenized_dataset['test'])

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=-1)

cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.show()

print(cm)